# CartPole playground

A place to poke at `CartPole-v1` before pointing the 2048 agent machinery at it. No learning code here — the last section is where a policy goes.

Kernel: **Python (NFL2026)**.

In [1]:
from typing import Callable, List, Optional, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation
from IPython.display import HTML

env = gym.make("CartPole-v1", render_mode="rgb_array")
print(env.observation_space)
print(env.action_space)

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Discrete(2)


## The state and action spaces

Observation is 4 floats:

| index | meaning | terminates outside |
| --- | --- | --- |
| 0 | cart position `x` | ±2.4 |
| 1 | cart velocity `x_dot` | — |
| 2 | pole angle `theta` (radians) | ±0.2095 (12°) |
| 3 | pole angular velocity `theta_dot` | — |

Actions: `0` = push cart left, `1` = push right. Reward is `+1` per surviving step, and the episode truncates at 500 steps — so 500 is a perfect score.

In [ ]:
from gymnasium.envs.classic_control.cartpole import 

In [12]:
type(env.env.env.env)

gymnasium.envs.classic_control.cartpole.CartPoleEnv

In [9]:
from gymnasium.wrappers.common import TimeLimit

In [13]:
obs, info = env.reset(seed=0)

In [16]:
obs, reward, terminated, truncated, _ = env.step(1)

In [21]:
obs

array([ 0.01323574,  0.17272775, -0.04686959, -0.3551522 ], dtype=float32)

## Stepping it by hand

Push right five times in a row and watch the pole tip over.

In [3]:
obs, _ = env.reset(seed=0)
for t in range(5):
    obs, reward, terminated, truncated, _ = env.step(1)
    print(f"t={t}  theta={obs[2]:+.4f}  x={obs[0]:+.4f}  r={reward}  term={terminated}")

t=0  theta=-0.0469  x=+0.0132  r=1.0  term=False
t=1  theta=-0.0540  x=+0.0167  r=1.0  term=False
t=2  theta=-0.0672  x=+0.0241  r=1.0  term=False
t=3  theta=-0.0866  x=+0.0353  r=1.0  term=False
t=4  theta=-0.1123  x=+0.0506  r=1.0  term=False


## Rollout helper

A policy is just `obs -> action`. Everything below runs through this.

In [4]:
Policy = Callable[[np.ndarray], int]


def rollout(
    policy: Policy,
    seed: Optional[int] = None,
    record_frames: bool = False,
) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Returns the observations visited (shape [T + 1, 4]) and, optionally, rendered frames."""
    obs, _ = env.reset(seed=seed)
    observations = [obs]
    frames = [env.render()] if record_frames else []
    while True:
        obs, _, terminated, truncated, _ = env.step(policy(obs))
        observations.append(obs)
        if record_frames:
            frames.append(env.render())
        if terminated or truncated:
            return np.array(observations), frames


def episode_lengths(policy: Policy, episodes: int = 100) -> np.ndarray:
    return np.array([len(rollout(policy, seed=i)[0]) - 1 for i in range(episodes)])


rng = np.random.default_rng(0)
random_policy: Policy = lambda obs: int(rng.integers(2))

observations, _ = rollout(random_policy, seed=0)
print(f"random policy survived {len(observations) - 1} steps")

random policy survived 18 steps


## Plotting an episode

The dashed lines are the termination thresholds.

In [ ]:
def plot_episode(observations: np.ndarray, title: str = "") -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, (col, label, limit) in zip(
        axes, [(2, "pole angle (rad)", 0.2095), (0, "cart position", 2.4)]
    ):
        ax.plot(observations[:, col])
        ax.axhline(limit, ls="--", c="r", lw=1)
        ax.axhline(-limit, ls="--", c="r", lw=1)
        ax.set_xlabel("step")
        ax.set_ylabel(label)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_episode(observations, "random policy")

## Watching a rollout

Renders the frames as an inline JS animation with playback controls.

In [ ]:
def animate(frames: List[np.ndarray], fps: int = 30) -> HTML:
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.axis("off")
    image = ax.imshow(frames[0])
    plt.close(fig)

    def draw(i: int) -> Tuple[plt.Artist, ...]:
        image.set_data(frames[i])
        return (image,)

    anim = animation.FuncAnimation(
        fig, draw, frames=len(frames), interval=1000 // fps, blit=True
    )
    return HTML(anim.to_jshtml())


_, frames = rollout(random_policy, seed=0, record_frames=True)
animate(frames)

## A hand-coded baseline

Push in whichever direction the pole is already falling, with a little damping on `theta_dot`. This is worth knowing before you train anything: it hits the 500-step cap on every seed, so CartPole is *solvable by a two-term linear rule*. A learned policy scoring 200 isn't doing well — the bar is 500.

In [ ]:
heuristic_policy: Policy = lambda obs: int(obs[2] + 0.5 * obs[3] > 0)

for name, policy in [("random", random_policy), ("heuristic", heuristic_policy)]:
    lengths = episode_lengths(policy)
    print(f"{name:<10} mean={lengths.mean():6.1f}  min={lengths.min():3d}  max={lengths.max():3d}")

plt.figure(figsize=(6, 3.5))
plt.hist(episode_lengths(random_policy), bins=20, alpha=0.6, label="random")
plt.hist(episode_lengths(heuristic_policy), bins=20, alpha=0.6, label="heuristic")
plt.xlabel("episode length")
plt.ylabel("episodes")
plt.legend()
plt.show()

In [ ]:
_, frames = rollout(heuristic_policy, seed=0, record_frames=True)
animate(frames)

## Your agent goes here

Anything matching `obs -> action` drops straight into `rollout`, `episode_lengths`, `plot_episode`, and `animate`. For a torch policy that's roughly:

```python
def policy(obs: np.ndarray) -> int:
    with torch.no_grad():
        logits = net(torch.as_tensor(obs, dtype=torch.float32))
    return int(torch.distributions.Categorical(logits=logits).sample())
```

Collecting into a `Trajectory` from `games.rl_2048.training.trajectory` needs the repo root on `sys.path`:

```python
import sys; sys.path.insert(0, "../../..")
from games.rl_2048.training.trajectory import Trajectory, TrajectoryStep
```